<a href="https://colab.research.google.com/github/anaclaracandidoo/projetodeep-learning/blob/main/07_dashboard_professora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install -q gradio

import gradio as gr

# CSS ajustado com estilo para os cards e formulários dos Planos de Aula
css = """
body, .gradio-container {
    background-color: #f0f4f7 !important;
    padding: 0 !important;
    margin: 0 !important;
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif !important;
}

.main-wrapper {
    display: flex;
    min-height: 100vh;
    width: 100%;
}

/* Sidebar Esquerda */
.sidebar-box {
    width: 250px;
    background-color: #1a424a;
    color: white;
    padding: 25px 15px;
    display: flex;
    flex-direction: column;
    flex-shrink: 0;
    min-height: 100vh;
}

.brand-title {
    display: flex;
    align-items: center;
    gap: 10px;
    font-size: 20px;
    font-weight: bold;
    margin-bottom: 35px;
    padding-left: 10px;
}

.menu-btn-custom {
    background-color: transparent !important;
    border: none !important;
    box-shadow: none !important;
    color: #a3c1c9 !important;
    text-align: left !important;
    font-size: 14px !important;
    padding: 12px 15px !important;
    margin-bottom: 8px !important;
    border-radius: 6px !important;
    cursor: pointer !important;
}

.menu-btn-custom:hover {
    background-color: #24525c !important;
    color: #ffffff !important;
}

.menu-btn-active {
    background-color: #275660 !important;
    color: #ffffff !important;
}

/* Área de Conteúdo */
.content-box {
    flex: 1;
    padding: 20px 40px;
    background-color: #f0f4f7;
}

.top-bar-flex {
    display: flex;
    justify-content: space-between;
    align-items: center;
    margin-bottom: 25px;
}

.header-title-text {
    font-size: 22px;
    font-weight: bold;
    color: #1a202c;
    margin-bottom: 25px;
}

/* Banner */
.banner-box {
    background-color: #dbe7f3;
    border-radius: 10px;
    padding: 20px 25px;
    display: flex;
    justify-content: space-between;
    align-items: center;
    margin-bottom: 25px;
}

.banner-box h3 {
    margin: 0 0 5px 0;
    color: #1e3a8a;
    font-size: 18px;
}

.banner-box p {
    margin: 0;
    color: #555770;
    font-size: 13px;
}

.badge-libras {
    background: white;
    padding: 8px 16px;
    border-radius: 20px;
    font-size: 12px;
    color: #333;
    display: flex;
    align-items: center;
    gap: 8px;
    box-shadow: 0 2px 4px rgba(0,0,0,0.03);
}

/* Cards de Ação */
.card-btn-custom {
    background: white !important;
    border: 1px solid #e5e7eb !important;
    border-radius: 12px !important;
    height: 140px !important;
    box-shadow: 0 2px 6px rgba(0,0,0,0.02) !important;
    display: flex !important;
    flex-direction: column !important;
    align-items: center !important;
    justify-content: center !important;
    font-weight: 600 !important;
    color: #1f2937 !important;
    font-size: 14px !important;
    cursor: pointer !important;
}

/* Cards de Histórico e Planos */
.historico-card {
    background: white;
    border-radius: 12px;
    border: 1px solid #e5e7eb;
    padding: 18px;
    margin-bottom: 12px;
    box-shadow: 0 2px 4px rgba(0,0,0,0.02);
}

.item-linha {
    padding: 12px 10px;
    border-bottom: 1px solid #f3f4f6;
    display: flex;
    justify-content: space-between;
    align-items: center;
    color: #374151;
    font-size: 14px;
}

.tag-ano {
    background-color: #eff6ff;
    color: #2563eb;
    padding: 4px 10px;
    border-radius: 12px;
    font-size: 12px;
    font-weight: 600;
}
"""

# --- FUNÇÕES DE NAVEGAÇÃO ---
def navegar_para(tela_destino):
    # Retorna [dash_visible, historico_visible, planos_visible]
    if tela_destino == "dashboard":
        return gr.update(visible=True), gr.update(visible=False), gr.update(visible=False)
    elif tela_destino == "historico":
        return gr.update(visible=False), gr.update(visible=True), gr.update(visible=False)
    elif tela_destino == "planos":
        return gr.update(visible=False), gr.update(visible=False), gr.update(visible=True)

# --- FUNÇÕES DO DASHBOARD & HISTÓRICO DE ATIVIDADES ---
def salvar_material(titulo_texto, ano_texto, historico_dados):
    if not titulo_texto:
        titulo_texto = "Atividade Adaptada Sem Título"
    if not ano_texto:
        ano_texto = "8º Ano"

    novo_item = {
        "id": len(historico_dados) + 1,
        "titulo": titulo_texto,
        "ano": ano_texto,
        "data": "Hoje",
        "conteudo": f"Material adaptado para o {ano_texto}."
    }
    historico_dados.insert(0, novo_item)

    html_dashboard = ""
    for item in historico_dados:
        html_dashboard += f"""
        <div class="item-linha">
            <div>📄 Prova / Atividade: <b>{item['titulo']}</b></div>
            <div class="tag-ano">{item['ano']}</div>
        </div>
        """

    return (
        gr.update(visible=False),
        gr.update(visible=True),
        html_dashboard,
        historico_dados,
        "",
        "",
        atualizar_lista_historico(historico_dados, "")
    )

def atualizar_lista_historico(historico_dados, termo_busca):
    if not historico_dados:
        return "<p style='color: #64748b; padding: 15px; background: white; border-radius: 8px;'>Nenhum material foi salvo ainda.</p>"

    html_out = ""
    termo = termo_busca.lower() if termo_busca else ""

    for item in historico_dados:
        if termo in item["titulo"].lower() or termo in item["ano"].lower():
            html_out += f"""
            <div class="historico-card">
                <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 8px;">
                    <span style="font-size: 16px; font-weight: bold; color: #1e293b;">📄 {item['titulo']}</span>
                    <span class="tag-ano">{item['ano']}</span>
                </div>
                <p style="font-size: 13px; color: #64748b; margin: 4px 0 12px 0;">{item['conteudo']}</p>
                <div style="font-size: 12px; color: #94a3b8; display: flex; justify-content: space-between;">
                    <span>📅 Criado em: {item['data']}</span>
                </div>
            </div>
            """

    if not html_out:
        html_out = "<p style='color: #64748b; padding: 15px; background: white; border-radius: 8px;'>Nenhuma atividade encontrada com esse termo.</p>"

    return html_out

def reutilizar_material(historico_dados):
    if len(historico_dados) > 0:
        ultimo = historico_dados[0]
        return gr.update(visible=True), f"Cópia de - {ultimo['titulo']}", ultimo['ano'], gr.update(visible=True), gr.update(visible=False), gr.update(visible=False)
    return gr.update(visible=True), "", "", gr.update(visible=True), gr.update(visible=False), gr.update(visible=False)


# --- FUNÇÕES DA TELA DE PLANOS DE AULA ---
def salvar_plano_aula(tema, ano, objetivo, metodologia, planos_dados):
    if not tema:
        tema = "Plano de Aula Sem Título"
    if not ano:
        ano = "8º Ano"

    novo_plano = {
        "tema": tema,
        "ano": ano,
        "objetivo": objetivo if objetivo else "Não informado.",
        "metodologia": metodologia if metodologia else "Adaptação visual e Libras.",
        "data": "Hoje"
    }

    planos_dados.insert(0, novo_plano)

    return (
        gr.update(visible=False), # oculta formulário
        planos_dados,
        "", "", "", "", # limpa campos de entrada
        atualizar_lista_planos(planos_dados, "") # atualiza HTML dos planos salvos
    )

def atualizar_lista_planos(planos_dados, termo_busca):
    if not planos_dados:
        return "<p style='color: #64748b; padding: 15px; background: white; border-radius: 8px;'>Nenhum plano de aula foi salvo ainda.</p>"

    html_out = ""
    termo = termo_busca.lower() if termo_busca else ""

    for plano in planos_dados:
        if termo in plano["tema"].lower() or termo in plano["ano"].lower():
            html_out += f"""
            <div class="historico-card">
                <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 8px;">
                    <span style="font-size: 16px; font-weight: bold; color: #1e3a8a;">📅 {plano['tema']}</span>
                    <span class="tag-ano">{plano['ano']}</span>
                </div>
                <div style="font-size: 13px; color: #334155; margin-bottom: 6px;">
                    <b>🎯 Objetivo:</b> {plano['objetivo']}
                </div>
                <div style="font-size: 13px; color: #334155; margin-bottom: 12px;">
                    <b>🤟 Metodologia / Acessibilidade:</b> {plano['metodologia']}
                </div>
                <div style="font-size: 12px; color: #94a3b8;">
                    <span>📅 Cadastrado em: {plano['data']}</span>
                </div>
            </div>
            """

    if not html_out:
        html_out = "<p style='color: #64748b; padding: 15px; background: white; border-radius: 8px;'>Nenhum plano de aula encontrado com esse termo.</p>"

    return html_out


with gr.Blocks(css=css, title="EduAcessível") as demo:

    # Estados de dados zerados por padrão
    historico_state = gr.State([])
    planos_state = gr.State([])

    with gr.Row(elem_classes="main-wrapper"):

        # --- SIDEBAR ESQUERDA ---
        with gr.Column(scale=1, min_width=250, elem_classes="sidebar-box"):
            gr.HTML("""
                <div class="brand-title">
                    <span>🎓🤟</span>
                    <span>EduAcessível</span>
                </div>
            """)

            btn_dash = gr.Button("🏠  Dashboard", elem_classes="menu-btn-custom menu-btn-active")
            btn_planos = gr.Button("📅  Planos de Aula", elem_classes="menu-btn-custom")
            btn_atividades = gr.Button("📝  Atividades", elem_classes="menu-btn-custom")
            btn_avaliacoes = gr.Button("📋  Avaliações Adaptadas", elem_classes="menu-btn-custom")
            btn_historico = gr.Button("⏱️  Histórico de Materiais", elem_classes="menu-btn-custom")

        # --- PAINEL PRINCIPAL ---
        with gr.Column(scale=4, elem_classes="content-box"):

            # Header Superior
            gr.HTML("""
                <div class="top-bar-flex">
                    <div style="font-size: 18px; color: #4a5568; cursor: pointer;">☰</div>
                    <div style="font-size: 14px; color: #2d3748; font-weight: 500;">
                        <span style="margin-right: 15px;">🔔</span>
                        <span>👤 Milena Belmiro Cândido ▾</span>
                    </div>
                </div>
            """)

            # ==========================================
            # TELA 1: DASHBOARD
            # ==========================================
            with gr.Column(visible=True) as tela_dashboard:
                gr.HTML('<div class="header-title-text">Painel da Professora</div>')

                # Banner
                gr.HTML("""
                    <div class="banner-box">
                        <div>
                            <h3>Bem-vinda, Milena!</h3>
                            <p>Adaptar os materiais de aprendizagem para alunos surdos.</p>
                        </div>
                        <div class="badge-libras">
                            <span>Acessibilidade libras</span>
                            <span style="font-size: 16px;">🤟 🖐️</span>
                        </div>
                    </div>
                """)

                # Cards de Ação Rápidas
                with gr.Row():
                    btn_card1 = gr.Button("➕ 📄\n\nCriar Nova Adaptação", elem_classes="card-btn-custom")
                    btn_card2 = gr.Button("🤟 ➔ 📑\n\nSimplificar Texto para Libras / Linguagem Visual", elem_classes="card-btn-custom")
                    btn_card3 = gr.Button("📷 🖼️ 💡\n\nAdicionar Imagens Ilustrativas", elem_classes="card-btn-custom")

                # Formulário de Cadastro de Material (Inicia Oculto)
                with gr.Column(visible=False) as area_formulario:
                    gr.HTML("<br><h3>Criar / Reutilizar Adaptação</h3>")
                    campo_titulo = gr.Textbox(label="Título do Material", placeholder="Ex: Prova de História")
                    campo_ano = gr.Textbox(label="Série/Ano", placeholder="Ex: 8º Ano")
                    btn_salvar = gr.Button("💾 Salvar Material", variant="primary")

                gr.HTML("<br>")

                # Lista de Recentes no Dashboard
                with gr.Column(visible=False, elem_classes="historico-card") as area_lista_dash:
                    gr.HTML("<h4>Atividades Salvas Recentemente</h4>")
                    html_lista_dash = gr.HTML("")

            # ==========================================
            # TELA 2: PLANOS DE AULA
            # ==========================================
            with gr.Column(visible=False) as tela_planos:
                gr.HTML('<div class="header-title-text">📅 Gestão de Planos de Aula</div>')

                with gr.Row():
                    btn_novo_plano = gr.Button("➕ Novo Plano de Aula", variant="primary", scale=1)
                    campo_busca_plano = gr.Textbox(label="🔍 Pesquisar Plano de Aula", placeholder="Digite o tema ou série...", scale=3)

                # Formulário para criar Plano de Aula (Inicia Oculto)
                with gr.Column(visible=False) as area_form_plano:
                    gr.HTML("<br><h3>Cadastrar Novo Plano de Aula</h3>")
                    plano_tema = gr.Textbox(label="Tema / Conteúdo da Aula", placeholder="Ex: Fotossíntese e Cadeia Alimentar")
                    plano_ano = gr.Textbox(label="Série/Ano", placeholder="Ex: 8º Ano")
                    plano_objetivo = gr.Textbox(label="Objetivo Pedagógico", placeholder="Ex: Compreender a transformação da luz em energia...", lines=2)
                    plano_metodologia = gr.Textbox(label="Estratégia de Acessibilidade em Libras", placeholder="Ex: Apresentação de vocabulário em Libras antes da leitura e mapas conceituais visuais.", lines=2)
                    btn_salvar_plano = gr.Button("💾 Salvar Plano de Aula", variant="primary")

                gr.HTML("<br>")

                # Lista de Planos de Aula Salvos (Inicia vazia)
                html_lista_planos = gr.HTML(atualizar_lista_planos([], ""))

            # ==========================================
            # TELA 3: HISTÓRICO DE ATIVIDADES
            # ==========================================
            with gr.Column(visible=False) as tela_historico:
                gr.HTML('<div class="header-title-text">⏱️ Histórico e Reutilização de Atividades</div>')

                with gr.Row():
                    campo_busca = gr.Textbox(label="🔍 Pesquisar Atividade Salva", placeholder="Digite o nome da matéria ou série...", scale=3)
                    btn_reutilizar_ultimo = gr.Button("🔄 Reutilizar Última Atividade", variant="secondary", scale=1)

                gr.HTML("<br>")

                html_lista_completa = gr.HTML(atualizar_lista_historico([], ""))

    # --- LÓGICA DE EVENTOS E INTERAÇÃO ---

    # Alternância entre as Telas na Sidebar
    btn_dash.click(fn=lambda: navegar_para("dashboard"), outputs=[tela_dashboard, tela_historico, tela_planos])
    btn_planos.click(fn=lambda: navegar_para("planos"), outputs=[tela_dashboard, tela_historico, tela_planos])
    btn_historico.click(fn=lambda: navegar_para("historico"), outputs=[tela_dashboard, tela_historico, tela_planos])

    # Eventos do Dashboard
    btn_card1.click(fn=lambda: gr.update(visible=True), outputs=area_formulario)
    btn_salvar.click(
        fn=salvar_material,
        inputs=[campo_titulo, campo_ano, historico_state],
        outputs=[area_formulario, area_lista_dash, html_lista_dash, historico_state, campo_titulo, campo_ano, html_lista_completa]
    )

    # Eventos do Histórico
    campo_busca.change(
        fn=atualizar_lista_historico,
        inputs=[historico_state, campo_busca],
        outputs=html_lista_completa
    )
    btn_reutilizar_ultimo.click(
        fn=reutilizar_material,
        inputs=historico_state,
        outputs=[area_formulario, campo_titulo, campo_ano, tela_dashboard, tela_historico, tela_planos]
    )

    # Eventos do Plano de Aula
    btn_novo_plano.click(fn=lambda: gr.update(visible=True), outputs=area_form_plano)
    btn_salvar_plano.click(
        fn=salvar_plano_aula,
        inputs=[plano_tema, plano_ano, plano_objetivo, plano_metodologia, planos_state],
        outputs=[area_form_plano, planos_state, plano_tema, plano_ano, plano_objetivo, plano_metodologia, html_lista_planos]
    )
    campo_busca_plano.change(
        fn=atualizar_lista_planos,
        inputs=[planos_state, campo_busca_plano],
        outputs=html_lista_planos
    )

# Executar aplicação
demo.launch(share=False)

/tmp/ipykernel_944/1108883598.py:304: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=css, title="EduAcessível") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>